# **Configuração inicial e carga dos dados**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Configurações estéticas padrão
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)

# Carregando a base de dados limpa
df = pd.read_csv("dados/water_potability_limpo.csv")


print(f" DATASET CARREGADO: {df.shape[0]} amostras e {df.shape[1]} atributos")

display(df.head())

# **Validação da Limpeza e Proporção do Alvo**


Aqui fazemos a auditoria para certificar que os valores ausentes sumiram e checamos o desbalanceamento estrutural original da base antes do SMOTE

In [ ]:
# Verificação de NaNs (Espera-se 0 após a imputação pela mediana por classe)
missing_data = df.isnull().sum().sum()
print(f"--> Total de dados ausentes restantes no dataset: {missing_data}")
if missing_data == 0:
    print("   [SUCESSO] O pipeline eliminou 100% dos valores nulos com segurança!")
else:
    print(df.isnull().sum())

# Proporção da Variável Alvo pós-limpeza
target_counts = df['Potability'].value_counts()
target_percent = df['Potability'].value_counts(normalize=True) * 100
print("\n--> Proporção da Variável Alvo (Desbalanceamento Estrutural):")
for idx, val in target_counts.items():
    print(f"   Classe {idx} (Potabilidade={idx}): {val} amostras ({target_percent[idx]:.2f}%)")

# **Estatística descritiva, assimetria e volatilidade**


Extrai o resumo descritivo tradicional, o cálculo de Skewness (assimetria residual pós-winsorização) e a volatilidade relativa por meio do Coeficiente de Variação (CV %).

In [ ]:
# Resumo estatístico geral
print("\n--> Resumo Geral (Média, Desvio Padrão e Quartis):")
display(df.describe().T.round(2))

# Medição de Skewness (Assimetria)
print("\n--> Coeficientes de Assimetria (Skewness):")
print(df.drop(columns=['Potability']).skew().round(4))

# Cálculo do Coeficiente de Variação (CV %) para medir a volatilidade relativa
print("\n--> Volatilidade Relativa (Coeficiente de Variação - CV %):")
cv = (df.drop(columns=['Potability']).std() / df.drop(columns=['Potability']).mean()) * 100
display(pd.DataFrame({'CV (%)': cv}).sort_values(by='CV (%)', ascending=False).round(2))

#### **Resumo geral(Média, Desvio Padrão e Quartis):**
Sucesso na Imputação: O campo count exibe exatamente 3.276 entradas para todas as variáveis. Isso comprova numericamente que a estratégia de preencher os valores nulos com a mediana por classe funcionou perfeitamente, sem perda de registros.

Preservação de Médias Físicas: O ph médio fixou-se em 7.07 (praticamente neutro), com intervalo restrito entre 3.89 e 10.26. Isso valida a correção feita no pipeline para eliminar leituras fisicamente impossíveis fora da escala tradicional $[0, 14]$.  

Variável-Alvo: A média de Potability é de 0.39, confirmando que cerca de 39% das amostras são de água potável. Isso detalha matematicamente o desbalanceamento estrutural da base antes da aplicação do SMOTE.

#### **Assimetria Residual (Skewness)**
Os coeficientes de assimetria revelam o impacto direto do tratamento de outliers via Winsorização.

Distribuições simétricas (Próximas a 0): A maioria absoluta das variáveis (ph, Hardness, Chloramines, Sulfate, Organic_carbon, Trihalomethanes e Turbidity) apresenta valores de Skewness extremamente baixos, flutuando na faixa de $-0.05$ a $+0.06$. Isso indica que a Winsorização por IQR limpou com maestria os valores extremos nas caudas, aproximando a distribuição dessas features de curvas perfeitamente simétricas (normais).  

Assimetria positiva moderada: Apenas Solids (0.4846) e Conductivity (0.2406) mantiveram uma leve assimetria à direita. Esse comportamento residual é esperado para índices de mineralização e condutividade em amostras ambientais, o que justifica e valida o uso do StandardScaler na etapa de pré-processamento. 

#### **Volatilidade Relativa (CV %)**

Como as variáveis possuem unidades de medida completamente distintas ($mg/L$, $ppm$, $NTU$), o Coeficiente de Variação é a única métrica justa para comparar a dispersão entre elas.

Alta Volatilidade (Solids = 39.13%): A concentração de sólidos dissolvidos totais é a métrica mais instável e dispersa do dataset. Isso sugere que o nível de mineralização da água varia bruscamente entre as fontes coletadas.

Estabilidade Rígida (Sulfate = 9.53%): O sulfato obteve o menor CV do conjunto, mantendo um comportamento altamente homogêneo e previsível ao longo de toda a base de dados.

Faixa Intermediária: Atributos críticos como o ph (19.54%) e Chloramines (21.68%) apresentam uma volatilidade moderada (próxima a 20%), indicando que variam dentro de limites toleráveis na natureza.


# **Análise gráfica univariada e bivariada**

Gera os gráficos principais: a densidade de distribuição do pH comparada com os limites teóricos da OMS e o boxplot de dureza modificado pela Winsorização

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Gráfico 1: Histograma e densidade Justa do pH pós-tratamento
sns.histplot(data=df, x='ph', hue='Potability', kde=True, ax=axes[0], palette='Set1', element='step', stat="density", common_norm=False)
axes[0].axvline(6.5, color='darkred', linestyle='--', linewidth=2, label='Limite Mín OMS (6.5)')
axes[0].axvline(8.5, color='darkred', linestyle='--', linewidth=2, label='Limite Máx OMS (8.5)')
axes[0].set_title('Distribuição de Densidade do pH (Dados Limpos) vs Potabilidade', fontsize=11, fontweight='bold')
axes[0].set_xlabel('Valor do pH[cite: 2]')
axes[0].legend()

# Gráfico 2: Boxplot de Hardness pós-tratamento de outliers
sns.boxplot(data=df, x='Potability', y='Hardness', ax=axes[1], palette='Set2', hue='Potability', legend=False)
axes[1].set_title('Distribuição de Dureza (Hardness pós-Winsorização) por Classe', fontsize=11, fontweight='bold')
axes[1].set_xlabel('Potabilidade (0 = Não Potável, 1 = Potável)')
axes[1].set_ylabel('Dureza (mg/L)')

plt.tight_layout()
plt.show()

### **Gráfico 1: Distribuição de Densidade do pH vs Potabilidade**

O Pico Central Notório (pH = 7.0): Observe a barra vertical massiva bem no centro do gráfico (em torno do valor 7.0). Esse pico acentuado é o reflexo direto da imputação por mediana feita pela sua colega. Como havia muitos valores ausentes originais de pH na base e eles foram preenchidos com a mediana da classe, a distribuição concentrou um grande volume de amostras exatamente nesse ponto central.  

Sobreposição Quase Total: As curvas de densidade azul (água potável) e vermelha (não potável) estão praticamente sobrepostas. Isso evidencia visualmente que o comportamento do pH não se altera entre águas seguras e inseguras.

O Paradoxo da OMS: As linhas tracejadas escuras marcam os limites recomendados pela OMS ($6.5$ a $8.5$). O gráfico prova de forma escancarada que existem centenas de amostras de água perfeitamente potáveis (em azul) localizadas bem abaixo de $6.5$ e bem acima de $8.5$. Isso é um excelente insight: mostra que o modelo preditivo não poderá se basear nas regras teóricas da OMS para classificar a água.

### **Gráfico 2: Distribuição de Dureza por Classe**

Efeito "Corte de Cabelo" nos Outliers: Olhe para as linhas horizontais que limitam as caudas (os "bigodes" do boxplot). Elas terminam de forma abrupta e achatada, acumulando pequenos pontinhos pretos exatamente nas extremidades (em torno de $120$ e $275$ mg/L). Isso é a assinatura visual clássica da Winsorização IQR. Em vez de excluir os dados ou deixar outliers extremos distorcerem a escala, a técnica limitou os valores atípicos estritamente às barreiras estatísticas estabelecidas.  

Simetria e Equivalência: As duas caixas (verde para não potável e laranja para potável) possuem tamanhos virtualmente idênticos, medianas alinhadas na faixa dos $196$ mg/L e dispersão equivalente. Isso reforça o diagnóstico anterior: a dureza da água, de forma isolada, também não possui poder de separação linear entre as duas classes.


# **Teste de hipótese inferencial**

Executa a inferência estatística matemática para testar se há diferença real nas distribuições das curvas de pH observadas no gráfico anterior.

In [ ]:
ph_potavel = df[df['Potability'] == 1]['ph']
ph_nao_potavel = df[df['Potability'] == 0]['ph']

stat, p_value = stats.mannwhitneyu(ph_potavel, ph_nao_potavel, alternative='two-sided')

print(f"Estatística U do Teste: {stat:.2f}")
print(f"p-valor calculado: {p_value:.5f}")

if p_value < 0.05:
    print("\n[RESULTADO]: Rejeitamos H0. Há diferença estatisticamente significativa entre as distribuições de pH.")
else:
    print("\n[RESULTADO]: Não rejeitamos H0. Não há evidência estatística de que o pH mude entre águas potáveis e não potáveis.")

#### **Teste de hipótese Mann-Whitney U:**

O Significado prático: Isso prova matematicamente que o preenchimento de dados feito na etapa de limpeza (imputação por mediana da classe) não gerou distorções artificiais que gerassem separabilidade na variável. As distribuições de pH entre as amostras de água potável e não potável permanecem estatisticamente equivalentes.

Impacto na modelagem: O pH isolado não possui poder discriminatório linear ou direto. Se tentássemos criar um classificador simples baseado apenas em regras condicionais de pH (ex: "se o pH estiver entre 6.5 e 8.5, classifique como potável"), o modelo falharia completamente, operando próximo ao acaso. 

#### **Matriz de Correlação Linear)**

In [ ]:
plt.figure(figsize=(6, 5))
correlation_matrix = df.corr()
mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))

sns.heatmap(correlation_matrix, mask=mask, annot=True, fmt=".2f", cmap='coolwarm', vmin=-1, vmax=1, linewidths=0.5)
plt.title("Matriz de Correlação Linear (Triângulo Inferior)", fontsize=13, fontweight='bold')
plt.show()

### **Matriz de correlação**

A análise do mapa de calor de Pearson nos revela duas conclusões estruturais fundamentais para o pipeline de Machine Learning da equipe:

**Inviabilidade de Modelos Lineares Puros:** A correlação linear direta de todos os atributos contra a variável `Potability` é estatisticamente nula (flutuando na faixa de $-0.03$ a $+0.03$). Isso demonstra empiricamente que classificadores que buscam linhas de separação simples (como a Regressão Logística) terão baixo desempenho preditivo se não houver transformações polinomiais ou uso de kernels não-lineares.

**Ortogonalidade e Independência das Features:** O maior coeficiente interno de correlação identificado foi de $-0.14$ (entre `Sulfate` e `Solids`). A ausência de multicolinearidade garante que os atributos explicativos são ortogonalmente independentes, permitindo que os algoritmos extraiam o máximo de informação útil de cada variável sem redundâncias no modelo.

**Justificativa Científica do Projeto:** Esses resultados validam perfeitamente a abordagem metodológica proposta para o projeto de realizar o benchmarking com algoritmos baseados em conjuntos de árvores de decisão (**Random Forest** e **XGBoost**). Como a separabilidade não é linear, estes modelos serão capazes de segmentar o espaço de características criando regras condicionais cruzadas multidimensionais altamente complexas.